enhance LLM with LoRA. we want Docnet style, concise version LLM.

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

## 1. 모델 로드 + LoRA 부착

Gemma 4 E2B는 멀티모달 모델이지만 우리는 **텍스트 전용** LoRA를 학습한다.
따라서 `FastVisionModel`이 아니라 **`FastModel`**을 쓰고, `finetune_vision_layers=False`로 비전 레이어를 얼려둔다.

In [ ]:
from unsloth import FastModel  # 텍스트 전용 LoRA는 FastVisionModel이 아니라 FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",
    max_seq_length = 1024,
    load_in_4bit = False,  # T4에서 OOM 나면 True로 (메모리 절약, 정밀도 소폭 하락)
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # 텍스트 전용이므로 비전 레이어는 동결
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 16,               # 데이터가 100건이라 32는 과함
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 2. 데이터 준비

AIris 도슨트 문답 데이터셋 `docent_seeds.jsonl` (100건)을 **Google Drive**에서 읽는다.

- 위치: `MyDrive/airis_refactoring/docent_seeds.jsonl`
- 출처: `art_metadata.json`(21,382점)의 실제 작품 기록에 근거해 작성
- 스타일: **관찰 유도형** — 관람객의 시선을 특정 지점으로 이끄는 도슨트 화법
- 구성: `artwork` 50 / `artist` 25 / `movement` 25
- 형식: `{"messages": [{"role": "user", ...}, {"role": "assistant", ...}]}` (이미지 없는 순수 텍스트)

> 매번 파일을 다시 올릴 필요가 없고, 런타임이 끊겨도 Drive에 그대로 남아 있다.
> 첫 실행 때 Drive 접근 권한을 묻는 팝업이 뜨면 승인할 것.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

DATA_PATH = "/content/drive/MyDrive/airis_refactoring/docent_seeds.jsonl"

# 경로가 조금 다를 수 있으니 못 찾으면 Drive 전체에서 같은 이름을 찾아본다
if not os.path.exists(DATA_PATH):
    hits = glob.glob("/content/drive/MyDrive/**/docent_seeds.jsonl", recursive=True)
    if hits:
        DATA_PATH = hits[0]
        print(f"지정 경로에 없어 자동으로 찾음: {DATA_PATH}")
    else:
        raise FileNotFoundError(
            "docent_seeds.jsonl을 Drive에서 못 찾음. "
            "왼쪽 파일 탐색기에서 실제 경로를 확인하고 DATA_PATH를 고칠 것."
        )

print(f"데이터 경로: {DATA_PATH}")

from datasets import load_dataset
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
print(dataset)

# 갈래별 분포 확인
import collections
print(collections.Counter(r["category"] for r in dataset["meta"]))

In [ ]:
dataset[0]

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

In [ ]:
from unsloth.chat_templates import standardize_data_formats

dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

# ⚠️ 반드시 아래 출력을 눈으로 확인할 것.
# 여기 찍히는 실제 마커(<start_of_turn>user / <start_of_turn>model 등)를
# 뒤의 train_on_responses_only 셀에 그대로 옮겨 적어야 한다.
print(dataset[0]["text"])

## 3. 학습 전 베이스라인

훈련 전 모델이 같은 질문에 어떻게 답하는지 미리 찍어둔다.
나중에 "좋아졌다"를 증명하려면 **대조군**이 필요하다.

In [ ]:
from transformers import TextStreamer

test_question = dataset[0]["messages"][0]["content"]
print(f"[질문] {test_question}\n[학습 전 답변]")

input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_question}],
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids, streamer = text_streamer, max_new_tokens = 256,
                   use_cache = True, temperature = 1.0, top_p = 0.95, top_k = 64)

## 4. 학습

- 데이터가 100건뿐이라 `max_steps` 대신 **`num_train_epochs=3`**을 쓴다. 한 바퀴로는 스타일이 안 붙는다.
- **`train_on_responses_only`**: 손실을 assistant 응답 토큰에만 계산한다. 질문 문장까지 예측하게 두면 학습이 낭비된다.

> ⚠️ 아래 `instruction_part` / `response_part`는 Gemma 계열 통상값이다.
> 위 셀에서 찍은 `print(dataset[0]["text"])` 출력과 **다르면 반드시 고칠 것**.
> 마커를 못 찾으면 에러 없이 조용히 전체 시퀀스에 손실이 걸린다 — 가장 눈치채기 어려운 실패다.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # 유효 배치 = 2 x 4 = 8
        max_grad_norm = 0.3,
        warmup_ratio = 0.03,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 1,
        save_strategy = "steps",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        max_length = 1024,
    ),
)

from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

In [ ]:
# @title 학습 전 메모리 상태
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title 학습 후 메모리 / 시간 통계
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## 5. 학습 후 추론

**학습 데이터에 없던 질문**으로 테스트한다. 그래야 특정 문답을 외운 게 아니라
말투가 일반화됐는지 확인할 수 있다.

Gemma 권장 추론 파라미터: `temperature=1.0`, `top_p=0.95`, `top_k=64`

In [ ]:
from transformers import TextStreamer

# 학습셋에 없는 질문들
for q in [
    "이 그림에서 뭘 눈여겨봐야 해?",
    "이 작품 배경이 왜 이렇게 어두워?",
    "인상주의가 뭐야?",
]:
    print(f"\n{'='*60}\n[질문] {q}\n[답변]")
    input_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")
    text_streamer = TextStreamer(tokenizer, skip_prompt = True)
    _ = model.generate(input_ids, streamer = text_streamer, max_new_tokens = 256,
                       use_cache = True, temperature = 1.0, top_p = 0.95, top_k = 64)

## 6. 저장

`save_pretrained`는 **LoRA 어댑터만** 저장한다(수십 MB). 원본 모델 없이는 못 쓴다.
독립적으로 굴러가는 모델이 필요하면 아래 병합 셀로 갈 것.

In [ ]:
model.save_pretrained("gemma_4_lora_docent")
tokenizer.save_pretrained("gemma_4_lora_docent")

# Hugging Face Hub에 올리려면:
# model.push_to_hub("your_name/gemma_4_lora_docent", token = "YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_name/gemma_4_lora_docent", token = "YOUR_HF_TOKEN")

## 7. 병합 및 내보내기

LoRA 어댑터는 그 자체로는 못 쓴다. 먼저 **원본 모델에 병합(merge)**해서 독립적인 체크포인트로 만든 뒤, `.litertlm`으로 변환한다.

```
LoRA 어댑터
    ↓ 병합
16bit HF 체크포인트
    ↓ litert-torch export_hf (int4)
.litertlm  (LiteRT-LM / LiteRtEngine)
```

`.litertlm` 변환은 Google이 만든 **`litert-torch`** 도구가 담당한다. Gemma 4는 **E2B / E4B가 공식 지원** 대상이다.

In [ ]:
# ── 공통 1단계: LoRA를 원본에 병합해 16bit HF 체크포인트 만들기 ──
# 두 포맷(.litertlm / .gguf) 모두 이 결과물에서 출발한다.

MERGED_DIR = "gemma_4_docent_16bit"

model.save_pretrained_merged(MERGED_DIR, tokenizer)
print(f"병합 완료: {MERGED_DIR}")

# Drive에도 복사해두면 런타임이 끊겨도 남는다
import shutil, os
DRIVE_MERGED = f"/content/drive/MyDrive/airis_refactoring/{MERGED_DIR}"
if not os.path.exists(DRIVE_MERGED):
    shutil.copytree(MERGED_DIR, DRIVE_MERGED)
    print(f"Drive에 복사: {DRIVE_MERGED}")

### 7-A. `.litertlm` 변환 (최종 목표)

Google의 `litert-torch` 도구로 HF 체크포인트 → `.litertlm` 변환한다.

- `--externalize_embedder` : 임베딩 레이어를 본체에서 분리 (온디바이스 메모리 절약)
- `--jinja_chat_template_override` : 채팅 템플릿을 litert-community 공식 것으로 맞춤

> ⚠️ **주의 두 가지**
> 1. 공식 문서 예시는 `--model`에 **HF Hub 경로**를 준다(`google/gemma-4-E2B-it` 같은). 로컬 디렉토리를 받는지는 문서에 명시가 없으니, 로컬로 실패하면 아래 Hub 업로드 경로를 쓸 것.
> 2. 문서에서 **양자화 플래그(int4/int8)를 찾지 못했다.** llama.cpp의 IQ4_NL과 공정 비교하려면 int4가 필요한데, 이건 변환 후 실제 파일 크기를 보고 판단해야 한다.

In [ ]:
# uv 설치 (litert-torch가 uv tool로 배포됨)
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]

!uv tool install litert-torch-nightly

In [ ]:
LITERT_OUT = "/content/gemma_4_docent_litertlm"

# 방법 1: 로컬 병합 체크포인트에서 바로 변환
!litert-torch export_hf \
  --model={MERGED_DIR} \
  --output_dir={LITERT_OUT} \
  --externalize_embedder \
  --jinja_chat_template_override=litert-community/gemma-4-E2B-it-litert-lm

# 위가 실패하면 → Hub에 올린 뒤 그 경로로 변환 (문서에 나온 방식)
#   model.push_to_hub_merged("YOUR_USERNAME/gemma-4-docent", tokenizer, token="YOUR_HF_TOKEN")
#   !litert-torch export_hf \
#     --model=YOUR_USERNAME/gemma-4-docent \
#     --output_dir={LITERT_OUT} \
#     --externalize_embedder \
#     --jinja_chat_template_override=litert-community/gemma-4-E2B-it-litert-lm

!ls -lh {LITERT_OUT}

In [ ]:
# 변환된 .litertlm이 실제로 도는지 Colab에서 먼저 확인 (폰에 올리기 전 검증)
!uv tool install litert-lm

!litert-lm run \
  {LITERT_OUT}/model.litertlm \
  --prompt="이 그림에서 뭘 눈여겨봐야 해?"

In [ ]:
# 완성된 .litertlm을 Drive로 회수 → 거기서 폰에 push
import shutil, os

DRIVE_LITERT = "/content/drive/MyDrive/airis_refactoring/gemma_4_docent_litertlm"
if os.path.exists(DRIVE_LITERT):
    shutil.rmtree(DRIVE_LITERT)
shutil.copytree(LITERT_OUT, DRIVE_LITERT)

print(f"Drive 저장 완료: {DRIVE_LITERT}")
!ls -lh {DRIVE_LITERT}

# 이후 컴퓨터에서 Drive 내려받아 폰에 push:
#   adb push model.litertlm /sdcard/Android/data/com.example.airis/files/
# 그리고 InferenceScreen.kt의 MODEL_FILE_NAME 상수를 이 파일명으로 변경